# Validating and Repairing a Model with pyshifty

This demo shows BuildingMOTIF's **algebraic validation report** and **template-guided graph repair**, powered by the [`pyshifty`](https://pypi.org/project/pyshifty/) SHACL engine. BuildingMOTIF uses pyshifty by default; this branch requires pyshifty 0.4.4.

When you validate with the `pyshifty` engine, BuildingMOTIF returns an `AlgebraicValidationContext` instead of the legacy `ValidationContext`. Rather than re-parsing a flattened W3C SHACL report, it consumes pyshifty's *algebraic* output directly and drives pyshifty's **symbolic repair** engine (abduction over the shape grammar).

Each failing focus node gets a **repair tree** of typed *holes*. The engine fills those holes from four candidate sources, then runs every candidate `ΔG` through pyshifty's **soundness gate** (it must not introduce a new violation):

1. **Recursive synthesis** — when a hole must conform to a sub-shape (`Hole.conforms_to_shapes`), the engine builds the value out structurally: reuse a node that already conforms, else mint one and recursively repair it against the sub-shape via `RepairSession.repair_node_against`. This materializes deep, correctly-typed values.
2. **Template reuse** — existing model nodes monomorphic to a BuildingMOTIF template (VF2 subgraph matching).
3. **Template mint** — a freshly grounded template instance, pulling in domain structure beyond the bare shape requirement.
4. **pyshifty native candidates** — flat reuse-first guesses, so we never do worse than stock pyshifty.

1. **Setup** — a BuildingMOTIF instance using the `pyshifty` engine
2. **Define** repair templates, shapes, and a deliberately broken model
3. **Algebraic validation** — inspect the `AlgebraicValidationContext` and its *witnesses*
4. **Repair proposals** — ranked, soundness-gated fixes from the four sources
5. **Recursive synthesis** — structurally-correct repair without any templates
6. **Apply a repair** and re-validate
7. **Lift repairs into BuildingMOTIF templates** (the `as_templates()` workflow)
8. **Bonus**: deletion-direction repair for `sh:not`

## 1. Setup

In [1]:
from rdflib import Namespace, Graph, Literal
from buildingmotif import BuildingMOTIF
from buildingmotif.dataclasses import Model, Library, AlgebraicValidationContext
from buildingmotif.namespaces import PARAM, BRICK, A

In [2]:
# Create a BuildingMOTIF instance backed by an in-memory database.
# pyshifty is the default engine and provides the algebraic validation + repair report.
building_motif = BuildingMOTIF("sqlite://")

## 2. Define repair templates, shapes, and a (broken) model

### A. A library of repair templates

Templates are BuildingMOTIF's domain vocabulary. The repair engine uses them as a candidate generator: it searches the model for subgraphs monomorphic to a template (to **reuse** existing nodes) and grounds templates to **mint** richer domain instances. Here we provide one template that builds a `brick:Temperature_Sensor`.

In [3]:
BLDG = Namespace("urn:bldg/")

repair_lib = Library.create("repair-templates")

sensor_body = Graph()
sensor_body.add((PARAM["name"], A, BRICK.Temperature_Sensor))
repair_lib.create_template("make-temperature-sensor", sensor_body)

sensor_body = Graph()

sensor_body.add((PARAM["name"], A, BRICK.Supply_Air_Temperature_Sensor))
repair_lib.create_template("make-sa-temperature-sensor", sensor_body)

print(sensor_body.serialize())

@prefix brick: <https://brickschema.org/schema/Brick#> .

<urn:___param___#name> a brick:Supply_Air_Temperature_Sensor .




### B. The shapes

Our requirement: every `brick:VAV` must have at least one `brick:hasPoint` relationship to a `brick:Temperature_Sensor`.

In [4]:
shapes = Graph().parse(data="""
@prefix sh:    <http://www.w3.org/ns/shacl#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl:   <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix :      <urn:shapes/> .

: a owl:Ontology .

brick:Supply_Air_Temperature_Sensor rdfs:subClassOf brick:Temperature_Sensor .

:vav-shape a sh:NodeShape ;
    sh:targetClass brick:VAV ;
    sh:property [
        sh:path brick:hasPoint ;
        sh:minCount 1 ;
        sh:class brick:Temperature_Sensor ;
    ] .
""", format="turtle")

shapes_lib = Library.from_ontology(shapes)

### C. A model with a deliberate violation

We create a single VAV with **no points** — it violates the shape above.

In [5]:
model = Model.create(BLDG)
model.add_triples((BLDG["vav1"], A, BRICK.VAV))

print(model.graph.serialize())

@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

<urn:bldg/> a owl:Ontology .

<urn:bldg/vav1> a brick:VAV .




## 3. Algebraic validation

Validating with the `pyshifty` engine returns an `AlgebraicValidationContext`. We pass `repair_libraries=[repair_lib]` so the engine can also use our templates when generating fixes.

In [6]:
ctx = model.validate(
    [shapes_lib.get_shape_collection()],
    repair_libraries=[repair_lib],
)

print("Report type:", type(ctx).__name__)
print("Model is valid:", ctx.valid)
print()
print(ctx.report_string)

Report type: AlgebraicValidationContext
Model is valid: False

Validation Report
Conforms: False

Violation result in urn:shapes/vav-shape (<urn:bldg/vav1>):
  Path: <https://brickschema.org/schema/Brick#hasPoint>
  Severity: Violation
  Value: <urn:bldg/vav1>
  Message: at least 1 value(s) required along <https://brickschema.org/schema/Brick#hasPoint>, found 0



### Read the violation horizon (witnesses)

The algebraic report exposes one `RepairWitness` per failing `(focus node, statement)`. Each witness explains *why* the node failed and carries its **repair tree** — an AND/OR/Repeat tree of typed holes describing every edit that would make the focus conform.

In [7]:
for witness in ctx.witnesses:
    print("Focus node:", witness.focus)
    print("  reason :", witness.reason())
    print("  blocked:", witness.is_blocked)
    print("  repair tree:")
    print("    " + witness.repair_tree.explain().replace("\n", "\n    "))

Focus node: urn:bldg/vav1
  reason : urn:bldg/vav1 CountLow on path <https://brickschema.org/schema/Brick#hasPoint> (have 0, need 1)
  blocked: False
  repair tree:
    Repeat [1..∞]:
      Edits:
        add <urn:bldg/vav1> <https://brickschema.org/schema/Brick#hasPoint> ?0
        ?0 : instance of <https://brickschema.org/schema/Brick#Temperature_Sensor>


## 4. Soundness-gated repair proposals

`witness.proposals()` returns ranked `RepairProposal`s. Each proposal's `ΔG` has been run through pyshifty's **gate**:

- `is_sound` — proved to introduce **no new** violation
- `is_progress` — proved to **remove** the violation
- `origin` — `synthesized` (recursive), `template:...`, or `pyshifty-candidate`
- `reused_nodes` — existing model nodes reused (found by `repair_node_against`/monomorphism)

Proposals are ranked sound + progress first, then by maximal reuse / minimal additions.

In [8]:
for witness in ctx.witnesses:
    print("Focus:", witness.focus)
    for p in witness.proposals():
        print(f"  [{p.origin:28s}] sound={p.is_sound!s:5} progress={p.is_progress!s:5} "
              f"additions={p.num_additions} reused={sorted(map(str, p.reused_nodes))}")
        for (s, pred, o) in p.additions:
            print(f"        + {s} {pred} {o}")
        for (s, pred, o) in p.deletions:
            print(f"        - {s} {pred} {o}")

Focus: urn:bldg/vav1
  [synthesized                 ] sound=True  progress=True  additions=2 reused=[]
        + urn:bldg/vav1 https://brickschema.org/schema/Brick#hasPoint urn:buildingmotif:repair#n1
        + urn:buildingmotif:repair#n1 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Temperature_Sensor
  [template:make-temperature-sensor] sound=True  progress=True  additions=2 reused=[]
        + urn:buildingmotif:repair#n2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Temperature_Sensor
        + urn:bldg/vav1 https://brickschema.org/schema/Brick#hasPoint urn:buildingmotif:repair#n2
  [template:make-sa-temperature-sensor] sound=True  progress=True  additions=2 reused=[]
        + urn:bldg/vav1 https://brickschema.org/schema/Brick#hasPoint urn:buildingmotif:repair#n3
        + urn:buildingmotif:repair#n3 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Supply_Air_Temperatur

**Read the ranking.** The top proposals **add a correctly *typed* `brick:Temperature_Sensor`** and so are sound *and* make progress — whether produced by recursive synthesis (`synthesized`) or grounded from our template (`template:make-temperature-sensor`). The `pyshifty-candidate` rows are flat reuse-first guesses drawn straight from the data graph; they are sound but make **no progress**, because binding the point to an arbitrary existing node does not satisfy the `sh:class` requirement.

## 5. Recursive synthesis works without templates

The `sh:class` requirement makes the hole a *ConformsTo* hole (`Hole.conforms_to_shapes`). The engine can therefore build a structurally-correct value on its own — minting a node and recursively repairing it against the sub-shape with `RepairSession.repair_node_against` — **even with no templates at all**.

In [9]:
ctx_no_templates = model.validate([shapes_lib.get_shape_collection()])  # no repair_libraries

best_synth = ctx_no_templates.witnesses[0].proposals()[0]
print("best origin :", best_synth.origin)
print("is_sound    :", best_synth.is_sound)
print("is_progress :", best_synth.is_progress)
for (s, p, o) in best_synth.additions:
    print("   +", s, p, o)

flat = [p for p in ctx_no_templates.witnesses[0].proposals() if p.origin == "pyshifty-candidate"]
print("\nFlat pyshifty candidates that make progress:", sum(p.is_progress for p in flat), "of", len(flat))

best origin : synthesized
is_sound    : True
is_progress : True
   + urn:bldg/vav1 https://brickschema.org/schema/Brick#hasPoint urn:buildingmotif:repair#n4
   + urn:buildingmotif:repair#n4 http://www.w3.org/1999/02/22-rdf-syntax-ns#type https://brickschema.org/schema/Brick#Temperature_Sensor

Flat pyshifty candidates that make progress: 0 of 5


### Deep recursion over `sh:node`

Recursive synthesis composes to arbitrary depth (bounded by a build-fuel budget). Here a node must reach — via a chain of properties — a value of a given class. The engine materializes the **whole chain** in one sound repair.

In [10]:
EX = Namespace("http://ex/")

deep_shapes = Graph().parse(data="""
@prefix sh:  <http://www.w3.org/ns/shacl#> .
@prefix ex:  <http://ex/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix :    <urn:shapes2/> .

: a owl:Ontology .

ex:Sub a sh:NodeShape ;
    sh:property [ sh:path ex:q ; sh:minCount 1 ; sh:class ex:Widget ] .

ex:Root-shape a sh:NodeShape ;
    sh:targetClass ex:Root ;
    sh:property [ sh:path ex:p ; sh:minCount 1 ; sh:node ex:Sub ] .
""", format="turtle")
deep_lib = Library.from_ontology(deep_shapes)

deep_model = Model.create(Namespace("urn:deep/"))
deep_model.add_triples((BLDG["root1"], A, EX.Root))

deep_ctx = deep_model.validate([deep_lib.get_shape_collection()])
fix = deep_ctx.witnesses[0].proposals()[0]
print("origin:", fix.origin, "| sound:", fix.is_sound, "| progress:", fix.is_progress)
print("synthesized chain:")
for (s, p, o) in fix.additions:
    print("   +", s, p, o)

origin: synthesized | sound: True | progress: True
synthesized chain:
   + urn:buildingmotif:repair#n6 http://ex/q urn:buildingmotif:repair#n7
   + urn:buildingmotif:repair#n7 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://ex/Widget
   + urn:bldg/root1 http://ex/p urn:buildingmotif:repair#n6


## 6. Apply a repair and re-validate

A proposal can be applied to the session to produce a patched graph. `RepairOutcome` records exactly what the fix does, and `advance()` gives us a fresh session over `G ⊕ ΔG` so we can confirm the violation is gone with nothing new introduced.

In [11]:
best = ctx.witnesses[0].proposals()[0]
print("Best proposal origin:", best.origin)
print("  violations fixed     :", len(best.outcome.fixed))
print("  violations introduced:", len(best.outcome.introduced))

patched_session = best.advance()
print("  remaining violations after applying the repair:", len(patched_session.witnesses()))

Best proposal origin: synthesized
  violations fixed     : 1
  violations introduced: 0
  remaining violations after applying the repair: 0


## 7. Lift repairs into BuildingMOTIF templates

`as_templates()` lifts the best sound repair per failure back into BuildingMOTIF `Template`s, so the repair plugs into the familiar "fill in the parameters" workflow. The focus node stays concrete; each freshly-minted individual becomes a parameter.

In [12]:
generated = ctx.as_templates()
for t in generated:
    print("-" * 70)
    print(t.body.serialize())
    print("parameters:", t.parameters)

----------------------------------------------------------------------
@prefix P: <urn:___param___#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .

<urn:bldg/vav1> brick:hasPoint P:repaired1 .

P:repaired1 a brick:Temperature_Sensor .


parameters: {'repaired1'}


In [13]:
# Fill the generated templates and add them to the model.
for t in generated:
    bindings = {param: BLDG[f"new_{param}"] for param in t.parameters}
    model.add_graph(t.substitute(bindings).to_graph())

# Re-validate: the model now conforms.
ctx_after = model.validate(
    [shapes_lib.get_shape_collection()],
    repair_libraries=[repair_lib],
)
print("Model is valid after repair:", ctx_after.valid)
print(model.graph.serialize())

Model is valid after repair: True
@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

<urn:bldg/> a owl:Ontology .

<urn:bldg/vav1> a brick:VAV ;
    brick:hasPoint <urn:bldg/new_repaired1> .

<urn:bldg/new_repaired1> a brick:Temperature_Sensor .




### Lifting *any* sound repair (not just the best)

`as_templates()` is opinionated: it keeps only the single best repair per failure. But every proposal the gate accepted is a valid fix, and you may prefer a different one (e.g. reuse over mint, or a particular template). Two methods give you the whole menu:

- `RepairProposal.as_template()` — lift *any* single proposal into a template (a fresh library is created if you don't pass one).
- `AlgebraicValidationContext.all_repair_templates()` — every sound repair, as separate templates, grouped by focus (the alternatives, un-merged). `RepairWitness.repair_templates()` does the same for one failure.

In [14]:
# Lift one specific alternative — say, the second-ranked sound repair — on its own.
second = ctx.witnesses[0].proposals()[1]
print("Lifting proposal:", second.origin)
print(second.as_template().body.serialize())

Lifting proposal: template:make-temperature-sensor
@prefix P: <urn:___param___#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .

<urn:bldg/vav1> brick:hasPoint P:repaired2 .

P:repaired2 a brick:Temperature_Sensor .




In [15]:
# Or get *all* sound repair alternatives, grouped by focus node.
alternatives = ctx.all_repair_templates()
for focus, templates in alternatives.items():
    print(f"{focus}: {len(templates)} alternative repair template(s)")
    for t in templates:
        print("  -", t.name, "->", ", ".join(sorted(t.body.objects())))

urn:bldg/vav1: 3 alternative repair template(s)
  - repair_vav1_2 -> https://brickschema.org/schema/Brick#Temperature_Sensor, urn:___param___#repaired3
  - repair_vav1_3 -> https://brickschema.org/schema/Brick#Temperature_Sensor, urn:___param___#repaired4
  - repair_vav1_4 -> https://brickschema.org/schema/Brick#Supply_Air_Temperature_Sensor, urn:___param___#repaired5


## 8. Bonus: deletion-direction repair (`sh:not`)

Not every fix is an addition. A violated `sh:not` constraint can only be repaired by **deletion** — and pyshifty's repair calculus handles that direction too, with the gate validating the deletion exactly the same way.

In [16]:
shapes_not = Graph().parse(data="""
@prefix sh:    <http://www.w3.org/ns/shacl#> .
@prefix brick: <https://brickschema.org/schema/Brick#> .
@prefix owl:   <http://www.w3.org/2002/07/owl#> .
@prefix :      <urn:shapes3/> .

: a owl:Ontology .

# A VAV must NOT be marked decommissioned
:no-decommissioned a sh:NodeShape ;
    sh:targetClass brick:VAV ;
    sh:not [ sh:path brick:status ; sh:hasValue "decommissioned" ] .
""", format="turtle")
shapes_not_lib = Library.from_ontology(shapes_not)

model_not = Model.create(Namespace("urn:bldg-not/"))
model_not.add_triples((BLDG["vav2"], A, BRICK.VAV))
model_not.add_triples((BLDG["vav2"], BRICK.status, Literal("decommissioned")))

ctx_not = model_not.validate([shapes_not_lib.get_shape_collection()])
fix = ctx_not.witnesses[0].proposals()[0]
print("sound:", fix.is_sound, "| progress:", fix.is_progress)
print("deletions proposed:")
for (s, p, o) in fix.deletions:
    print(f"  - {s} {p} {o}")
print("additions proposed:", fix.num_additions)

sound: True | progress: True
deletions proposed:
  - urn:bldg/vav2 https://brickschema.org/schema/Brick#status decommissioned
additions proposed: 0


## Summary

With the `pyshifty` engine, BuildingMOTIF's validation report becomes an `AlgebraicValidationContext` that:

- exposes failures as **witnesses** with structured **repair trees** computed by abduction over the shape algebra (not a flattened report);
- repairs *ConformsTo* holes by **recursive synthesis** — building deep, correctly-typed values via `repair_node_against`, with no templates required;
- additionally uses your **templates** via **VF2 monomorphism** to reuse existing model nodes and pull in domain structure;
- **gates every candidate** for soundness, so a repair never trades one violation for another;
- supports **addition and deletion** directions; and
- still **lifts repairs into BuildingMOTIF templates** via `as_templates()` for the existing workflow.